### Extract

#### WTI Oil Prices
* Lag features
#### Economic growth
* US dollar index 
* S&P 500 index 
* US Federal Reserve rate
* Volatility index (VIX)
* Consumer price index (CPI).

source: https://www.investopedia.com/articles/investing/072515/top-factors-reports-affect-price-oil.asp


EIA
* wti_prices -> daily
* oil_production -> monthly
* input_utilization -> weekly
* gasoline_price -> weekly
* imports_and_exports -> weekly
* weekly_stocks -> weekly

FRED
* us_dollar_index -> daily
* volatility_index -> daily
* cpi_energy -> monthly
* s&p500 -> daily

#### Energy Information Administration (EIA) Api
| Features | Frequence | Data Until |
|---|---|---|
| WTI | Daily | up-to-date |
| Oil Production | Monthly | July 2025|
|Weekly Input Utilization | Weekly | up-to-date |
| Gasoline Price | Weekly | up-to-date |
|Imports & Exports | Weekly | up-to-date |
|Crude Oil Supplied | Monthly | July 2025 |

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
%pwd

'/workspaces/oil-optimization'

In [ ]:
from dotenv import dotenv_values

secrets = dotenv_values('.env')

api_calls  = {
    'wti_prices':{
        'url':'https://api.eia.gov/v2/petroleum/pri/spt/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPCWTI',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'frequency':'daily',
                    'data[0]':'value'
                }
    },
    'oil_production':{
        'url':'https://api.eia.gov/v2/petroleum/crd/crpdn/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                },
        'date_intervals':[
                    ('2015-01','2020-02'),
                    ('2020-02','2025-03'),
                    ('2025-03','')
                    ]
    },
    'input_utilization':{
        'url':'https://api.eia.gov/v2/petroleum/pnp/wiup/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                }
    },
    'gasoline_price':{
        'url':'https://api.eia.gov/v2/petroleum/pri/gnd/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPM0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value',
                },
        'date_intervals':[
                    ('2015-01-01','2017-12-31'),
                    ('2018-01-01','2021-03-31'),
                    ('2021-04-01','2024-06-30'),
                    ('2024-07-01',None)
                    ]
    },
    'imports_and_exports':{
        'url':'https://api.eia.gov/v2/petroleum/move/wkly/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                },
        'date_intervals':[
                    ('2015-01-01','2023-08-31'),
                    ('2023-09-01','')
                    ]
    },
    'oil_supplied':{
        'url':'https://api.eia.gov/v2/petroleum/cons/psup/data',
        'payload':{
                    'api_key':secrets['EIA_API_KEY'],
                    'facets[product][]':'EPC0',
                    'start':'2015-01-01',
                    'sort[0][column]':'period',
                    'sort[0][direction]':'asc',
                    'data[0]':'value'
                }
    }}

In [ ]:
from src.oil_optimization.utils.io_helpers import read_yaml

config = read_yaml('config/config.yml')
api_config = read_yaml('config/api_config.yml')
eia_api_calls = api_config['eia_api']

data_dir = config['data_ingestion']['data_dir']

In [ ]:
import logging
import sys
from abc import ABC, abstractmethod
from typing import Any
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from dotenv import dotenv_values
from src.oil_optimization.utils.io_helpers import read_yaml

SECRETS = dotenv_values('.env')
EIA_API_KEY = SECRETS['EIA_API_KEY']
FRED_API_KEY = SECRETS['FRED_API_KEY']

logging.basicConfig(
                    level=logging.INFO,
                    format="%(asctime)s | %(name)s | %(levelname)s | %(lineno)d | %(message)s",
                    handlers=[
                    logging.FileHandler("myapp.log"),
                    logging.StreamHandler(sys.stdout),
                    ],
                    force=True
                    )
logger = logging.getLogger(__name__)


class BaseExtractor(ABC):
    def __init__(self):
        super().__init__()
        self.config = read_yaml("config/config.yml")
        self.data_dir = self.config["data_ingestion"]["data_dir"]
        self.api_config = read_yaml("config/api_config.yml")
        self.session = requests.Session()

        adapter = HTTPAdapter(max_retries=3)
        self.session.mount("https://", adapter)

    def get(self, url: str, method: str = "GET", payload: dict[str,Any] = {}):
        try:
            if not payload:
                logger.warning("No query parameters are being sent to %s due to empty payload", url)
            
            r = self.session.request(method=method,
                                    url=url,
                                    timeout=20,
                                    params=payload)
            logger.info("Status code: %s", r.status_code)
            r.raise_for_status()
            time.sleep(3)

            if r.status_code == 200:
                return r.json()
        except requests.exceptions.HTTPError as e:
            logger.exception(e)
    
    @abstractmethod
    def extract_data(self, response_data):
        pass

    @abstractmethod
    def create_file(self, label, params):
        pass

    def save_to_csv(self, df: pd.DataFrame, filename: str):
        path = f'{self.data_dir}/raw/{filename}.csv'
        df.to_csv(path, index=False)
        logger.info("%s.csv successfully created!", filename)

    def close(self):
        self.session.close()
        logger.info("Session Closed.")

class EIAExtractor(BaseExtractor):
    def __init__(self, api_key: str | None = None) -> None:
        super().__init__()
        self.eia_api_config = self.api_config['eia_api']
        self.api_key = api_key

        logger.info("============================ Starting HTTP Session for EIA API ============================")

    def extract_data(self, response_data):
        return response_data['response']['data']

    def create_file(self, label:str, params:dict[str,Any]):
        data_list = []
        params['payload']['api_key'] = self.api_key
        if "date_intervals" in params.keys():
            for i, date in enumerate(params['date_intervals']):
                payload = params['payload'].copy()
                payload['start'] = date[0]

                if date[1]:
                    payload['end'] = date[1]
                if i == 0:
                    logger.info("Limited data retrieval of %s from %s due to API limits, starting pagination...", label ,params["url"])
                logger.info("Sending HTTP request #%s", i+1)
                data = self.get(params["url"], payload=payload)
                json_data = self.extract_data(data)

                data_list += json_data
                
            df = pd.DataFrame(data_list)
            self.save_to_csv(df, label)

        else:
            data = self.get(params['url'], payload=params['payload'])
            json_data = self.extract_data(data)
            df = pd.DataFrame(json_data)
            self.save_to_csv(df, label)

class FREDExtractor(BaseExtractor):
    def __init__(self, api_key: str | None = None):
        super().__init__()
        self.fred_api_config = self.api_config['fred_api']
        self.api_key = api_key
        self.url = self.fred_api_config["url"]
        self.payload_dicts = self.fred_api_config["payload"]

        logger.info("============================ Starting HTTP Session for FRED API ============================")

    def extract_data(self, response_data):
        return response_data['observations']

    def create_file(self, label: str, params: dict[str, Any]):
        params["api_key"] = self.api_key

        logger.info("Sending HTTP request for %s", label)
        data = self.get(url=self.url, payload=params)
        json_data = self.extract_data(data)
        df = pd.DataFrame(json_data).rename({"date":"period"}, axis=1)
        self.save_to_csv(df, label)

if __name__ == "__main__":
    eia_extractor = EIAExtractor(api_key=EIA_API_KEY)
    for key, params_dict in eia_extractor.eia_api_config.items():
        eia_extractor.create_file(label=key, params=params_dict)
    eia_extractor.close()

    fred_extractor = FREDExtractor(api_key=FRED_API_KEY)
    for key, params_dict in fred_extractor.payload_dicts.items():
        fred_extractor.create_file(label=key, params=params_dict)
    fred_extractor.close()

#### Federeal Reserve Bank of St. Louis (FRED)
* US Dollar Index (DXY)
* Volatility Index (VIX)
* Consumer Price Index (CPI)

In [6]:
from oil_optimization.utils.io_helpers import read_yaml
from dotenv import dotenv_values

secrets = dotenv_values('.env')

fred_api = read_yaml('config/api_config.yml')['fred_api']

In [72]:
url = f'https://api.stlouisfed.org/fred/series/observations?series_id={'DTWEXBGS'}&api_key={secrets['FRED_API_KEY']}&file_type=json&observation_start=2015-01-01'

r = requests.get(url=url)


In [73]:
list_data = []
for item in r.json()['observations']:
    list_data.append({'date':item['date'],
           'value':item['value']})

In [74]:
pd.DataFrame(list_data)

,date,value
0,2015-01-01,.
1,2015-01-02,102.9027
2,2015-01-05,103.4976
3,2015-01-06,103.2938
4,2015-01-07,103.6316
...,...,...
2812,2025-10-13,.
2813,2025-10-14,121.5815
2814,2025-10-15,121.2669
2815,2025-10-16,121.0834


In [ ]:
import os
from dotenv import dotenv_values
import pandas as pd
os.chdir('..')

SECRETS = dotenv_values('.env')
EIA_API_KEY = SECRETS['EIA_API_KEY']
FRED_API_KEY = SECRETS['FRED_API_KEY']

In [ ]:
from src.oil_optimization.data_pipeline.extractor import EIAExtractor, FREDExtractor
from dotenv import dotenv_values
import pandas as pd

SECRETS = dotenv_values('.env')
EIA_API_KEY = SECRETS['EIA_API_KEY']
FRED_API_KEY = SECRETS['FRED_API_KEY']

eia_extractor = EIAExtractor()
for key, params_dict in eia_extractor.eia_api_config.items():
    eia_extractor.create_file(label=key, params=params_dict)

fred_extractor = FREDExtractor()
for key, params_dict in fred_extractor.fred_api_config.items():
    params_dict['api_key'] = FRED_API_KEY
    fred_json = fred_extractor.get(url=fred_extractor.url, payload=params_dict)
    fred_data = fred_extractor.extract_data(fred_json)
    dataframe = pd.DataFrame(fred_data). \
        rename({'date':'period'},axis=1) # Change of date name from 'date' to 'period'
    fred_extractor.save_to_csv(dataframe, key)

ModuleNotFoundError: No module named 'src'

### Data Quality (GX)

In [1]:
import great_expectations as gx
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
context = gx.get_context(mode="file", project_root_dir="./data_quality")

In [5]:
context.data_sources.add_pandas(name="etl_source")

PandasDatasource(type='pandas', name='etl_source', id=UUID('8d4405b5-c316-4a75-9a26-f4bb0683f4c1'), assets=[])

In [4]:
data_source = context.data_sources.get("etl_source")

In [13]:
data_source.add_dataframe_asset(name="daily_asset")
data_source.add_dataframe_asset(name="weekly_asset")
data_source.add_dataframe_asset(name="monthly_asset")

DataFrameAsset(name='monthly_asset', type='dataframe', id=UUID('5fc0f49b-8c58-444c-a968-e50fa1c423df'), order_by=[], batch_metadata={}, batch_definitions=[])

In [5]:
daily_asset = data_source.get_asset(name="daily_asset")
weekly_asset = data_source.get_asset(name="weekly_asset")
monthly_asset = data_source.get_asset(name="monthly_asset")

In [16]:
daily_asset.add_batch_definition_whole_dataframe(name="wti_prices")
daily_asset.add_batch_definition_whole_dataframe(name="us_dollar_index")
daily_asset.add_batch_definition_whole_dataframe(name="volatility_index")
daily_asset.add_batch_definition_whole_dataframe(name="sp500")

BatchDefinition(id=UUID('212f3846-4d83-49aa-a12d-b1fd63427164'), name='sp500', partitioner=None)

In [22]:
monthly_asset.add_batch_definition_whole_dataframe(name="oil_production")
monthly_asset.add_batch_definition_whole_dataframe(name="cpi_energy")

BatchDefinition(id=UUID('bcd7c9c1-a493-4f10-92a5-126e0d5d0503'), name='cpi_energy', partitioner=None)

In [23]:
weekly_asset.add_batch_definition_whole_dataframe(name="input_utilization")
weekly_asset.add_batch_definition_whole_dataframe(name="gasoline_price")
weekly_asset.add_batch_definition_whole_dataframe(name="imports_and_exports")
weekly_asset.add_batch_definition_whole_dataframe(name="weekly_stocks")

BatchDefinition(id=UUID('2aa4af4d-a4c4-43e8-85af-e8b00e6a88cf'), name='weekly_stocks', partitioner=None)

In [18]:
daily_asset.batch_definitions

[BatchDefinition(id=UUID('1c7e6786-6203-4e78-86b3-90c954f14bdb'), name='wti_prices', partitioner=None),
 BatchDefinition(id=UUID('a3a23f99-2c4a-4070-b93e-27d1e20c47fb'), name='us_dollar_index', partitioner=None),
 BatchDefinition(id=UUID('2042a499-1f55-4dcd-abbe-4964c80a0f7f'), name='volatility_index', partitioner=None),
 BatchDefinition(id=UUID('212f3846-4d83-49aa-a12d-b1fd63427164'), name='sp500', partitioner=None)]

In [9]:
import os
import pandas as pd

os.listdir("data/raw")
files = ['sp500.csv',
 'input_utilization.csv',
 'us_dollar_index.csv',
 'imports_and_exports.csv',
 'cpi_energy.csv',
 'oil_production.csv',
 'wti_prices.csv',
 'weekly_stocks.csv',
 'gasoline_price.csv',
 'volatility_index.csv']

df = pd.read_csv(f"data/raw/{files[1]}")

In [23]:
import great_expectations.expectations as gxe

In [13]:
suite = gx.ExpectationSuite(name="input_utilization_suite")

In [17]:
suite = context.suites.get(name="input_utilization_suite")

In [24]:
gxe.ExpectColumnDistinctValuesToBeInSet(column="area-name",
                                        value_set=['PADD 1', 'PADD 2', 'PADD 3', 'PADD 4', 'PADD 5', 'U.S.'])

ExpectColumnDistinctValuesToBeInSet(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=False, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='area-name', row_condition=None, condition_parser=None, value_set=['PADD 1', 'PADD 2', 'PADD 3', 'PADD 4', 'PADD 5', 'U.S.'])

In [32]:
df["units"].unique()

array(['MBBL/D'], dtype=object)

In [35]:
unique_expectation = gxe.ExpectColumnDistinctValuesToBeInSet(column="area-name",
                                        value_set=['PADD 1', 'PADD 2', 'PADD 3', 'PADD 4', 'PADD 5', 'U.S.'])
gtzero_expectation = gxe.ExpectColumnValuesToBeBetween(column="value", min_value=0)
notnull_expecation = gxe.ExpectColumnValuesToNotBeNull(column="value")
unit_expectation = gxe.ExpectColumnDistinctValuesToEqualSet(column="units", value_set=["MBBL/D"])

In [38]:
suite.add_expectation(unique_expectation)
suite.add_expectation(gtzero_expectation)
suite.add_expectation(notnull_expecation)
suite.add_expectation(unit_expectation)

RuntimeError: Cannot add Expectation because it already belongs to an ExpectationSuite. If you want to update an existing Expectation, please call Expectation.save(). If you are copying this Expectation to a new ExpectationSuite, please copy it first (the core expectations and some others support copy(expectation)) and set `Expectation.id = None`.

In [54]:
batch_definition = data_source.get_asset(name="weekly_asset").get_batch_definition("input_utilization")
validation_definition = gx.ValidationDefinition(name="input_utilization_validation", suite=suite, data=batch_definition)

In [55]:
context.validation_definitions.add(validation_definition)

ValidationDefinition(name='input_utilization_validation', data=BatchDefinition(id='01f06e01-193d-42d4-b30b-c205659c6627', name='input_utilization', partitioner=None), suite={
  "name": "input_utilization_suite",
  "id": "d2518a58-2e9f-4013-af90-28bd48622f10",
  "expectations": [
    {
      "type": "expect_column_distinct_values_to_be_in_set",
      "kwargs": {
        "column": "area-name",
        "value_set": [
          "PADD 1",
          "PADD 2",
          "PADD 3",
          "PADD 4",
          "PADD 5",
          "U.S."
        ]
      },
      "meta": {},
      "id": "2736e897-d3e0-4109-825e-98ac0882d0bf",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_be_between",
      "kwargs": {
        "column": "value",
        "min_value": 0.0
      },
      "meta": {},
      "id": "23111449-a9bf-4fec-adb7-791f76d5eb38",
      "severity": "critical"
    },
    {
      "type": "expect_column_values_to_not_be_null",
      "kwargs": {
        "column": "v

In [46]:
import datetime as dt
dt.datetime.now().strftime("%Y-%m-%d")

'2026-04-05'